# DATA WRANGLING NHL PROJECT
## BAIS:3250 Data Wrangling
### Jacob Johnsen, Lucas Durflinger, Landon Doering
#### 04/29/2026
---

## File Library
---

- `nhl_merged.csv`


<br>

## Import Library
---

In [1]:
import pandas as pd


<br>

# Load, Drop, and Clean Column Names
---

In [2]:
#load merged NHL dataset
merged_df = pd.read_csv("nhl_merged.csv")


In [3]:
#drop unnecessary index column if it exists
merged_df = merged_df.drop(columns=["Unnamed: 0",
                                  "game_date_api",
                                  "game_state_api",
                                  "game_date",
                                  "season_api"],
                                  errors="ignore"
)

#individual columns
team_stat_cols = [
    "team_id",
    "teamName",
    "abbreviation",
    "won",
    "goals",
    "shots",
    "hits",
    "pim",
    "powerPlayOpportunities",
    "powerPlayGoals",
    "faceOffWinPercentage",
    "giveaways",
    "takeaways",
    "blocked"
]

#shared columns
game_info_cols = [
    "game_id",
    "season",
    "date_time_CT",
    "venue_api",
    "game_type_api",
    "playoff_title_api",
    "playoff_round_api"
]

#separate home and away rows
home_df = merged_df[merged_df["HoA"] == "home"][game_info_cols + team_stat_cols].copy()
away_df = merged_df[merged_df["HoA"] == "away"][["game_id"] + team_stat_cols].copy()

#rename team columns to 'home_xxx' or 'away_xxx'
home_df = home_df.rename(columns={col: f"home_{col}" for col in team_stat_cols})
away_df = away_df.rename(columns={col: f"away_{col}" for col in team_stat_cols})

#merge home and away dataframes into one row per game
all_games = pd.merge(home_df, away_df, on="game_id", how="inner")

#check
print(all_games.shape)
all_games.head(30)


(4007, 35)


,game_id,season,date_time_CT,venue_api,game_type_api,playoff_title_api,playoff_round_api,home_team_id,home_teamName,home_abbreviation,...,away_goals,away_shots,away_hits,away_pim,away_powerPlayOpportunities,away_powerPlayGoals,away_faceOffWinPercentage,away_giveaways,away_takeaways,away_blocked
0,2015020001,20152016,2015-10-07 18:00:00-05:00,Air Canada Centre,2.0,NaN,NaN,10,Maple Leafs,TOR,...,3.0,29.0,26.0,8.0,1.0,0.0,46.9,9.0,6.0,15.0
1,2015020002,20152016,2015-10-07 19:00:00-05:00,United Center,2.0,NaN,NaN,16,Blackhawks,CHI,...,3.0,27.0,43.0,4.0,0.0,0.0,47.4,13.0,10.0,15.0
2,2015020003,20152016,2015-10-07 21:00:00-05:00,Scotiabank Saddledome,2.0,NaN,NaN,20,Flames,CGY,...,5.0,44.0,16.0,36.0,3.0,0.0,47.8,4.0,5.0,13.0
3,2015020004,20152016,2015-10-07 21:30:00-05:00,STAPLES Center,2.0,NaN,NaN,26,Kings,LAK,...,5.0,32.0,19.0,24.0,8.0,2.0,48.6,2.0,1.0,13.0
4,2015020005,20152016,2015-10-08 18:00:00-05:00,TD Garden,2.0,NaN,NaN,6,Bruins,BOS,...,6.0,32.0,32.0,6.0,0.0,0.0,50.8,8.0,7.0,13.0
5,2015020006,20152016,2015-10-08 18:00:00-05:00,First Niagara Center,2.0,NaN,NaN,7,Sabres,BUF,...,3.0,22.0,22.0,6.0,0.0,0.0,58.3,3.0,4.0,13.0
6,2015020007,20152016,2015-10-08 18:30:00-05:00,Amalie Arena,2.0,NaN,NaN,14,Lightning,TBL,...,2.0,25.0,26.0,6.0,3.0,1.0,50.8,8.0,6.0,6.0
7,2015020008,20152016,2015-10-08 19:00:00-05:00,Scottrade Center,2.0,NaN,NaN,19,Blues,STL,...,1.0,25.0,14.0,13.0,3.0,1.0,35.8,4.0,3.0,13.0
8,2015020009,20152016,2015-10-08 19:00:00-05:00,Bridgestone Arena,2.0,NaN,NaN,18,Predators,NSH,...,1.0,26.0,11.0,6.0,3.0,0.0,64.7,6.0,5.0,9.0
9,2015020010,20152016,2015-10-08 19:30:00-05:00,American Airlines Center,2.0,NaN,NaN,25,Stars,DAL,...,0.0,37.0,21.0,10.0,3.0,0.0,51.3,5.0,11.0,16.0


In [4]:
#remove 'xxx_api' suffix off columns that still have it
all_games = all_games.rename(
    columns={col: col.replace("_api", "") for col in all_games.columns}
)


In [5]:
#move to excel for external check
all_games.to_excel("Whole_Season.xlsx")


<br>

# Create Season DataFrame, Regular Season DataFrame, and Playoffs DataFrame
---

In [6]:
#entire season dataframe
whole_season_df = all_games.copy()

#remove game_type NaNs from whole_season_df
whole_season_df = whole_season_df.dropna(subset=['game_type'])

#regular season games only
regular_season_df = all_games[all_games["game_type"] == 2.0].copy()

#drop playoff_title and playoff_round because playoffs don't happen in regular season
regular_season_df = regular_season_df.drop(columns = {"playoff_title", "playoff_round"}, errors = 'ignore')

#playoff games only
playoff_df = all_games[all_games["game_type"] == 3.0].copy()

#check the shape of each dataframe
print("Whole season:", whole_season_df.shape)
print("Regular season:", regular_season_df.shape)
print("Playoffs:", playoff_df.shape)


Whole season: (3993, 35)
Regular season: (3731, 33)
Playoffs: (262, 35)


- All rows in `all_games["game_type"]` that have a NaN value must be dropped as they are series games. For example, Team A and
  Team B are competing in a Best-of-7 series. Team A wins the series in 5 games. Game 6 and game 7 are still in the dataset,
  but were never played. Almost all of the columns for these excess games are NaN besides where they were meant to be played,
  and between the teams involved in the series. These are already filtered out in `regular_season_df` and `playoff_df`, but not in
  `whole_season_df`. 

<br>

## Display Each DataFrame to Check for Errors in Cleaning
---

In [7]:
#display whole_season_df
print("whole_season_df:")
display(whole_season_df.head(10))


whole_season_df:


,game_id,season,date_time_CT,venue,game_type,playoff_title,playoff_round,home_team_id,home_teamName,home_abbreviation,...,away_goals,away_shots,away_hits,away_pim,away_powerPlayOpportunities,away_powerPlayGoals,away_faceOffWinPercentage,away_giveaways,away_takeaways,away_blocked
0,2015020001,20152016,2015-10-07 18:00:00-05:00,Air Canada Centre,2.0,NaN,NaN,10,Maple Leafs,TOR,...,3.0,29.0,26.0,8.0,1.0,0.0,46.9,9.0,6.0,15.0
1,2015020002,20152016,2015-10-07 19:00:00-05:00,United Center,2.0,NaN,NaN,16,Blackhawks,CHI,...,3.0,27.0,43.0,4.0,0.0,0.0,47.4,13.0,10.0,15.0
2,2015020003,20152016,2015-10-07 21:00:00-05:00,Scotiabank Saddledome,2.0,NaN,NaN,20,Flames,CGY,...,5.0,44.0,16.0,36.0,3.0,0.0,47.8,4.0,5.0,13.0
3,2015020004,20152016,2015-10-07 21:30:00-05:00,STAPLES Center,2.0,NaN,NaN,26,Kings,LAK,...,5.0,32.0,19.0,24.0,8.0,2.0,48.6,2.0,1.0,13.0
4,2015020005,20152016,2015-10-08 18:00:00-05:00,TD Garden,2.0,NaN,NaN,6,Bruins,BOS,...,6.0,32.0,32.0,6.0,0.0,0.0,50.8,8.0,7.0,13.0
5,2015020006,20152016,2015-10-08 18:00:00-05:00,First Niagara Center,2.0,NaN,NaN,7,Sabres,BUF,...,3.0,22.0,22.0,6.0,0.0,0.0,58.3,3.0,4.0,13.0
6,2015020007,20152016,2015-10-08 18:30:00-05:00,Amalie Arena,2.0,NaN,NaN,14,Lightning,TBL,...,2.0,25.0,26.0,6.0,3.0,1.0,50.8,8.0,6.0,6.0
7,2015020008,20152016,2015-10-08 19:00:00-05:00,Scottrade Center,2.0,NaN,NaN,19,Blues,STL,...,1.0,25.0,14.0,13.0,3.0,1.0,35.8,4.0,3.0,13.0
8,2015020009,20152016,2015-10-08 19:00:00-05:00,Bridgestone Arena,2.0,NaN,NaN,18,Predators,NSH,...,1.0,26.0,11.0,6.0,3.0,0.0,64.7,6.0,5.0,9.0
9,2015020010,20152016,2015-10-08 19:30:00-05:00,American Airlines Center,2.0,NaN,NaN,25,Stars,DAL,...,0.0,37.0,21.0,10.0,3.0,0.0,51.3,5.0,11.0,16.0


In [8]:
#display regular_season_df
print("regular_season_df:")
display(regular_season_df.head(10))


regular_season_df:


,game_id,season,date_time_CT,venue,game_type,home_team_id,home_teamName,home_abbreviation,home_won,home_goals,...,away_goals,away_shots,away_hits,away_pim,away_powerPlayOpportunities,away_powerPlayGoals,away_faceOffWinPercentage,away_giveaways,away_takeaways,away_blocked
0,2015020001,20152016,2015-10-07 18:00:00-05:00,Air Canada Centre,2.0,10,Maple Leafs,TOR,False,1.0,...,3.0,29.0,26.0,8.0,1.0,0.0,46.9,9.0,6.0,15.0
1,2015020002,20152016,2015-10-07 19:00:00-05:00,United Center,2.0,16,Blackhawks,CHI,False,2.0,...,3.0,27.0,43.0,4.0,0.0,0.0,47.4,13.0,10.0,15.0
2,2015020003,20152016,2015-10-07 21:00:00-05:00,Scotiabank Saddledome,2.0,20,Flames,CGY,False,1.0,...,5.0,44.0,16.0,36.0,3.0,0.0,47.8,4.0,5.0,13.0
3,2015020004,20152016,2015-10-07 21:30:00-05:00,STAPLES Center,2.0,26,Kings,LAK,False,1.0,...,5.0,32.0,19.0,24.0,8.0,2.0,48.6,2.0,1.0,13.0
4,2015020005,20152016,2015-10-08 18:00:00-05:00,TD Garden,2.0,6,Bruins,BOS,False,2.0,...,6.0,32.0,32.0,6.0,0.0,0.0,50.8,8.0,7.0,13.0
5,2015020006,20152016,2015-10-08 18:00:00-05:00,First Niagara Center,2.0,7,Sabres,BUF,False,1.0,...,3.0,22.0,22.0,6.0,0.0,0.0,58.3,3.0,4.0,13.0
6,2015020007,20152016,2015-10-08 18:30:00-05:00,Amalie Arena,2.0,14,Lightning,TBL,True,3.0,...,2.0,25.0,26.0,6.0,3.0,1.0,50.8,8.0,6.0,6.0
7,2015020008,20152016,2015-10-08 19:00:00-05:00,Scottrade Center,2.0,19,Blues,STL,True,3.0,...,1.0,25.0,14.0,13.0,3.0,1.0,35.8,4.0,3.0,13.0
8,2015020009,20152016,2015-10-08 19:00:00-05:00,Bridgestone Arena,2.0,18,Predators,NSH,True,2.0,...,1.0,26.0,11.0,6.0,3.0,0.0,64.7,6.0,5.0,9.0
9,2015020010,20152016,2015-10-08 19:30:00-05:00,American Airlines Center,2.0,25,Stars,DAL,True,3.0,...,0.0,37.0,21.0,10.0,3.0,0.0,51.3,5.0,11.0,16.0


In [9]:
#display playoff_df
print("playoff_df:")
display(playoff_df.head(10))


playoff_df:


,game_id,season,date_time_CT,venue,game_type,playoff_title,playoff_round,home_team_id,home_teamName,home_abbreviation,...,away_goals,away_shots,away_hits,away_pim,away_powerPlayOpportunities,away_powerPlayGoals,away_faceOffWinPercentage,away_giveaways,away_takeaways,away_blocked
1230,2015030121,20152016,2016-04-13 18:00:00-05:00,Amalie Arena,3.0,1st Round,1.0,14,Lightning,TBL,...,2.0,36.0,35.0,18.0,5.0,0.0,44.6,4.0,8.0,10.0
1231,2015030141,20152016,2016-04-13 19:00:00-05:00,CONSOL Energy Center,3.0,1st Round,1.0,5,Penguins,PIT,...,2.0,37.0,50.0,10.0,5.0,1.0,48.5,2.0,3.0,10.0
1232,2015030161,20152016,2016-04-13 20:30:00-05:00,Scottrade Center,3.0,1st Round,1.0,19,Blues,STL,...,0.0,35.0,24.0,8.0,5.0,0.0,45.8,4.0,7.0,22.0
1233,2015030131,20152016,2016-04-14 18:00:00-05:00,Verizon Center,3.0,1st Round,1.0,15,Capitals,WSH,...,0.0,19.0,26.0,35.0,4.0,0.0,54.4,9.0,7.0,21.0
1234,2015030111,20152016,2016-04-14 19:00:00-05:00,BB&T Center,3.0,1st Round,1.0,13,Panthers,FLA,...,5.0,26.0,40.0,8.0,2.0,1.0,51.5,10.0,2.0,18.0
1235,2015030151,20152016,2016-04-14 20:30:00-05:00,American Airlines Center,3.0,1st Round,1.0,25,Stars,DAL,...,0.0,22.0,28.0,12.0,2.0,0.0,44.9,3.0,3.0,21.0
1236,2015030181,20152016,2016-04-14 21:30:00-05:00,STAPLES Center,3.0,1st Round,1.0,26,Kings,LAK,...,4.0,23.0,32.0,6.0,4.0,1.0,54.2,8.0,2.0,15.0
1237,2015030122,20152016,2016-04-15 18:00:00-05:00,Amalie Arena,3.0,1st Round,1.0,14,Lightning,TBL,...,2.0,32.0,30.0,68.0,5.0,1.0,51.4,9.0,5.0,14.0
1238,2015030112,20152016,2016-04-15 18:30:00-05:00,BB&T Center,3.0,1st Round,1.0,13,Panthers,FLA,...,1.0,42.0,32.0,8.0,3.0,0.0,42.9,9.0,6.0,8.0
1239,2015030162,20152016,2016-04-15 19:00:00-05:00,Scottrade Center,3.0,1st Round,1.0,19,Blues,STL,...,3.0,29.0,25.0,6.0,3.0,1.0,56.2,2.0,3.0,14.0


<br>

## Convert Each Cleaned DataFrame to 'xxx.csv' Files
---

In [10]:
#convert each dataframe to .csv
whole_season_df.to_csv("whole_season_games.csv", encoding = 'utf-8', sep = ',', index=False)
regular_season_df.to_csv("regular_season_games.csv", encoding = 'utf-8', sep = ',', index=False)
playoff_df.to_csv("playoff_games.csv", encoding = 'utf-8', sep = ',', index=False)


<br>